# Phase 2 — Modeling, Explainability, Fairness & Drift (results)

This notebook narrates and visualises the outputs of the Phase-2 pipeline
(`python -m bixi.pipeline`). It reads the artifacts a completed run wrote to S3
(`metrics.json`, `fairness_report.json`, `drift_summary.json`, SHAP images) and
the MLflow tracking server. Set `RUN_ID` / `TARGET` to a finished run.

See `docs/phase2_modeling.md` for the design and decisions.

In [ ]:
import os, sys, json
sys.path.insert(0, '../src')
from bixi import config, io

RUN_ID = os.getenv('BIXI_RUN_ID', 'cloud-2024')
TARGET = 'departure'
def stage_key(stage, name):
    return f"{config.stage_prefix(RUN_ID, TARGET, stage)}/{name}"
print('pipeline bucket:', config.PIPELINE_BUCKET, '| run:', RUN_ID, '| target:', TARGET)

## 1. Model selection & performance (vs. naive baseline)

In [ ]:
import pandas as pd
m = io.get_json(stage_key('train', 'metrics.json'))
print('Selected model:', m['selected']['name'])
print('Baseline (val):', m['baseline_val'])
cand = pd.DataFrame({k: v for k, v in m['candidates'].items()}).T[['rmse','mae','r2']]
display(cand.sort_values('rmse'))
pd.DataFrame({'val': m['selected']['val'], 'test': m['selected']['test']})

## 2. Explainability (SHAP)

In [ ]:
from IPython.display import Image, display
import tempfile
for img in ['shap_importance_bar.png', 'shap_summary_beeswarm.png']:
    p = os.path.join(tempfile.gettempdir(), img)
    io.download_file(stage_key('explain', img), p)
    display(Image(filename=p))

## 3. Fairness — error parity across demand tiers & geographic zones

In [ ]:
fr = io.get_json(stage_key('fairness', 'fairness_report.json'))
print('Flags:', fr['flags'])
print('tier RMSE disparity ratio:', fr.get('tier_rmse_disparity_ratio'))
display(pd.DataFrame(fr['by_demand_tier']))

## 4. Drift (4 types) — with the data caveat

All four drift types are computed against the 2024 baseline. **Caveat:** 2024 is
the baseline for all years and 2025 historical features derive from matching 2024
periods, which limits engineered-feature drift; this is analysis under data
constraints, not a live monitor (no weekly cron). The Evidently HTML reports are
in S3 under the `drift/` stage prefix.

In [ ]:
ds = io.get_json(stage_key('drift', 'drift_summary.json'))
for period, s in ds.items():
    fd = s['feature_drift']
    print(f"{period}: feat_drifted={fd['n_drifted']}/{fd['n_features']} "
          f"target_drift={s['target_drift']['drift']} "
          f"concept_alert={s['concept_drift']['concept_drift_alert']} "
          f"(current_r2={s['concept_drift']['current_r2']:.3f})")

## 5. MLflow

Experiments (`bixi-demand-departure`, `bixi-demand-arrival`) hold every candidate,
FLAML and Optuna run; the best model per target is registered with the
`production` alias. Open the tracking UI at the `BixiMlflow` stack's `MlflowPublicUrl`.